# Strands Agent with Traceloop Observability via OpenLLMetry on Amazon Bedrock AgentCore Runtime

## Overview

This notebook shows how to deploy a Strands agent to Amazon Bedrock AgentCore Runtime and instrument it with Traceloop using OpenLLMetry. The example uses Bedrock Claude models and emits telemetry to Traceloop through the OpenTelemetry protocol exposed by OpenLLMetry.

## Key Components

- **Strands Agents**: Python framework for multi-tool agents with telemetry hooks
- **Amazon Bedrock AgentCore Runtime**: Managed runtime for hosting agents with secure endpoints
- **OpenLLMetry**: Traceloop's OpenTelemetry-based instrumentation for LLM apps
- **Traceloop**: Observability platform that processes OpenLLMetry traces for debugging and analytics

## Architecture

The agent container runs on AgentCore Runtime. OpenLLMetry captures workflow and task spans, forwarding them to Traceloop via OTLP. A lazy initialization flow ensures the telemetry exporter is configured before the Strands agent is constructed.

## Prerequisites

- Python 3.10+
- AWS credentials with Bedrock and AgentCore permissions
- Traceloop account with API key
- Docker installed locally
- Access to Amazon Bedrock Claude models in `us-west-2`



## OpenTelemetry Destinations

OpenLLMetry forwards spans using standard OpenTelemetry protocols, so you can stream data to any compatible backend. Popular options include [Traceloop](https://traceloop.com/docs/openllmetry/integrations/introduction), Datadog, Grafana Tempo, Honeycomb, Splunk, and more—use the same instrumentation shown here and just update the OTLP endpoint and headers for your observability sink.


## Installation

Install the dependencies listed in `requirements.txt`.



In [ ]:
%pip install --force-reinstall -U -r requirements.txt --quiet


## Agent Implementation

The agent file `strands_claude_traceloop.py` defines a travel assistant that logs workflow and tool spans through OpenLLMetry before invoking Bedrock.



In [ ]:
%%writefile strands_claude_traceloop.py
import os
import logging
from functools import lru_cache
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent, tool
from strands.models import BedrockModel
from traceloop.sdk import Traceloop
from traceloop.sdk.decorators import workflow, task
from ddgs import DDGS

logging.basicConfig(level=logging.ERROR, format="[%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)
logger.setLevel(os.getenv("AGENT_RUNTIME_LOG_LEVEL", "INFO").upper())

app = BedrockAgentCoreApp()


def _init_traceloop():
    Traceloop.init(app_name="strands-traceloop-agent", disable_batch=True)


@task(name="tool.web_search")
@tool
def web_search(query: str) -> str:
    try:
        ddgs = DDGS()
        results = ddgs.text(query, max_results=5)
        formatted_results = []
        for i, result in enumerate(results, 1):
            formatted_results.append(
                f"{i}. {result.get('title', 'No title')}\n"
                f"   {result.get('body', 'No summary')}\n"
                f"   Source: {result.get('href', 'No URL')}\n"
            )
        return "\n".join(formatted_results) if formatted_results else "No results found."
    except Exception as exc:
        return f"Error searching the web: {exc}"


def _bedrock_model() -> BedrockModel:
    region = os.getenv("AWS_DEFAULT_REGION", "us-west-2")
    model_id = os.getenv("BEDROCK_MODEL_ID", "us.anthropic.claude-3-7-sonnet-20250219-v1:0")
    return BedrockModel(model_id=model_id, region_name=region, temperature=0.0, max_tokens=1024)


@app.entrypoint
@workflow(name="strands_travel_agent")
def strands_agent_bedrock(payload, context=None) -> str:
    agent = _agent()
    user_input = payload.get("prompt")
    logger.info("[%s] User input: %s", getattr(context, "session_id", "unknown"), user_input)
    response = agent(user_input)
    return response.message["content"][0]["text"]


@lru_cache(maxsize=1)
def _agent() -> Agent:
    _init_traceloop()
    return Agent(model=_bedrock_model(), system_prompt=_system_prompt(), tools=[web_search])


def _system_prompt() -> str:
    return (
        "You are an experienced travel agent specializing in personalized travel recommendations "
        "with access to recent web information. Provide recommendations with current context and "
        "concise planning details."
    )


if __name__ == "__main__":
    app.run()


### Configure AgentCore Runtime deployment

Use the starter toolkit to configure the runtime image, execution role, and entrypoint. Disable the default OTEL setup so OpenLLMetry can manage telemetry.



In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "strands_traceloop_observability"

response = agentcore_runtime.configure(
    entrypoint="strands_claude_traceloop.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
    disable_otel=True,
)
response


## Deploy to AgentCore Runtime

Launch the agent container to AgentCore Runtime once the configuration is ready.



In [ ]:
# Traceloop configuration
traceloop_endpoint = "https://api.traceloop.com"
traceloop_api_key = "<traceloop-api-key>"

# The header value must encode the space between Bearer and the key as %20
traceloop_headers = f"Authorization=Bearer%20{traceloop_api_key}"

launch_result = agentcore_runtime.launch(
    env_vars={
        "BEDROCK_MODEL_ID": "us.anthropic.claude-3-7-sonnet-20250219-v1:0",
        "TRACELOOP_BASE_URL": traceloop_endpoint,
        "TRACELOOP_HEADERS": traceloop_headers,
        "TRACELOOP_APP_NAME": "strands-traceloop-agent",
        "TRACELOOP_DISABLE_BATCH": "true",
        "DISABLE_ADOT_OBSERVABILITY": "true",
    }
)
launch_result


## Check Deployment Status

Wait for the runtime endpoint to reach a terminal status before invoking.



In [ ]:
import time

status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]
terminal_states = {"READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"}

while status not in terminal_states:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]
    print(status)

print(f"Final status: {status}")


## Invoke AgentCore Runtime

Send a prompt payload to validate the deployment.



In [ ]:
invoke_response = agentcore_runtime.invoke({
    "prompt": "Plan a long weekend in Denver with outdoor activities and local dining options."
})


In [ ]:
from IPython.display import Markdown, display

display(Markdown("".join(invoke_response["response"])))


## View Traces in Traceloop

1. Open the Traceloop dashboard and choose the project connected to your API key.
2. Inspect the workflow and task spans for agent invocations and tool usage recorded by OpenLLMetry.

